# Appendix B: Behind the Data -- The Data Generating Process

**Causal Inference: A Pokemon Approach -- Kanto Region Edition**

---

Every dataset in this textbook was created by *structural equations* -- mathematical functions that encode the true causal relationships between variables. This appendix pulls back the curtain and shows you **exactly how the data was made**.

Why does this matter?

1. **You know the truth.** Because we wrote the DGP, we know every causal effect, every confounding path, and every selection mechanism. This lets you check whether your estimator recovers the truth.
2. **You can break things on purpose.** Change a coefficient, add a confounder, violate an assumption -- and watch what happens to your estimates.
3. **Reproducibility.** The single global seed (`np.random.default_rng(151)` -- one Mew to rule them all) means you get the same data every time.

---

## 1. Setup & Imports

In [ ]:
import sys
from pathlib import Path

# Ensure project root is on the path
PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT / "src") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "src"))
if str(PROJECT_ROOT / "data" / "dgp") not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT / "data" / "dgp"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Import the DGP module directly
import structural_equations as dgp

# Import our plotting utilities
from kanto_utils import apply_kanto_theme, oak_says, nurse_joy_says, badge_earned
from kanto_utils import pokemon_scatter, type_color, type_colors_dict

apply_kanto_theme()

print(f"DGP module loaded from: {dgp.__file__}")
print(f"Global RNG seed: 151 (the original Kanto Pokedex!)")

## 2. The Kanto Trainers Dataset: A Complete Walkthrough

The `kanto_trainers` dataset is the workhorse of this textbook -- 2,000 trainers, 42 variables, and a rich causal structure. Let's generate it fresh and inspect the structural equations.

### 2.1 Generating the data

**Important:** The DGP uses a *shared global RNG*. To reproduce the shipped datasets, you must call the generators in the same order as `generate_all.py`. Here, we reset the RNG for isolated demonstrations.

In [ ]:
# Reset the global RNG so this notebook is self-contained
dgp.RNG = np.random.default_rng(151)

trainers = dgp.generate_kanto_trainers(n=2000)
print(f"Shape: {trainers.shape}")
print(f"Columns ({trainers.shape[1]}): {list(trainers.columns)}")
trainers.head()

### 2.2 Structural Equations -- The Causal Skeleton

The DGP starts with three **latent** (unobserved) variables that drive everything downstream:

| Latent Variable | Distribution | Role |
|:----------------|:-------------|:-----|
| `patience` | Uniform(0, 100) | Drives safari visits, strategy |
| `natural_talent` | Uniform(0, 100) | Drives IVs, strategy |
| `dedication` | Uniform(0, 100) | Drives play hours, items |

Then **wealth** is determined by hometown (with noise):

```
wealth_bonus = +0.3 if hometown in {Celadon, Saffron} else -0.3 if {Lavender, Pallet} else 0
wealth = clip(round(Normal(3 + bonus, 0.8)), 1, 5)
```

**Starter type** is chosen via a softmax over three utilities:

```
u_grass  = 0.0 + 0.05 * (experience - 4)
u_fire   = 0.1 + 0.15 * (wealth - 3)       <-- wealthier trainers prefer Fire
u_water  = -0.05 + 0.12 * (experience - 4)  <-- experienced trainers prefer Water
P(starter = type) = softmax(u_grass, u_fire, u_water)
```

**Badges** (the core outcome) are driven by a logistic aggregation:

```
badge_logit = -4.0 + 0.04*strategy + 0.03*team_level + 0.15*type_diversity
              + 0.01*tm_count + 0.005*spending/100 + 0.3*exp_share + 0.01*held_items
badges = clip(floor(logistic(badge_logit) * 9 + noise), 0, 8)
```

In [ ]:
oak_says(
    "Every variable in this dataset exists for a reason. The <b>structural equations</b> "
    "define the <em>true</em> causal mechanism. When you run an estimator, you're trying "
    "to recover what we already know from the DGP. That's the beauty of simulation -- "
    "you have a ground truth to check against."
)

### 2.3 Visualising Key Causal Relationships

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))

# 1. Wealth -> Dept store spending
ax = axes[0, 0]
for w in sorted(trainers['wealth'].unique()):
    subset = trainers[trainers['wealth'] == w]
    ax.hist(subset['dept_store_spending'], bins=30, alpha=0.5, label=f'Wealth={w}')
ax.set_xlabel('Dept Store Spending')
ax.set_ylabel('Count')
ax.set_title('Wealth -> Spending\n(Direct causal effect)')
ax.legend(fontsize=8)

# 2. Experience -> Strategy score
ax = axes[0, 1]
ax.scatter(trainers['trainer_experience'], trainers['strategy_score'],
           alpha=0.2, s=10, color='#3B4CCA')
# Add a LOESS-like trend via binning
bins = pd.cut(trainers['trainer_experience'], bins=15)
means = trainers.groupby(bins, observed=True)['strategy_score'].mean()
bin_centers = [b.mid for b in means.index]
ax.plot(bin_centers, means.values, 'o-', color='#EE1515', linewidth=2, markersize=5)
ax.set_xlabel('Trainer Experience (years)')
ax.set_ylabel('Strategy Score')
ax.set_title('Experience -> Strategy\n(Causal + patience confound)')

# 3. Wealth -> Starter type (selection bias!)
ax = axes[0, 2]
ct = pd.crosstab(trainers['wealth'], trainers['starter_type'], normalize='index')
ct.plot(kind='bar', stacked=True, ax=ax,
        color=[type_color('grass'), type_color('fire'), type_color('water')])
ax.set_xlabel('Wealth Level')
ax.set_ylabel('Proportion')
ax.set_title('Wealth -> Starter Choice\n(Non-random selection)')
ax.legend(title='Starter', fontsize=8)
ax.tick_params(axis='x', rotation=0)

# 4. Strategy -> Badges
ax = axes[1, 0]
ax.scatter(trainers['strategy_score'], trainers['badges'],
           alpha=0.15, s=10, color='#4DAD5B')
bins_s = pd.cut(trainers['strategy_score'], bins=15)
means_s = trainers.groupby(bins_s, observed=True)['badges'].mean()
bin_centers_s = [b.mid for b in means_s.index]
ax.plot(bin_centers_s, means_s.values, 'o-', color='#EE1515', linewidth=2, markersize=5)
ax.set_xlabel('Strategy Score')
ax.set_ylabel('Badges Earned')
ax.set_title('Strategy -> Badges\n(Causal pathway)')

# 5. Team Level -> Badges
ax = axes[1, 1]
ax.scatter(trainers['team_level_avg'], trainers['badges'],
           alpha=0.15, s=10, color='#F7D02C')
bins_t = pd.cut(trainers['team_level_avg'], bins=15)
means_t = trainers.groupby(bins_t, observed=True)['badges'].mean()
bin_centers_t = [b.mid for b in means_t.index]
ax.plot(bin_centers_t, means_t.values, 'o-', color='#EE1515', linewidth=2, markersize=5)
ax.set_xlabel('Team Level (avg)')
ax.set_ylabel('Badges Earned')
ax.set_title('Team Level -> Badges\n(Causal pathway)')

# 6. Latent patience -> Observable strategy (hidden confounder revealed!)
ax = axes[1, 2]
ax.scatter(trainers['patience'], trainers['strategy_score'],
           alpha=0.15, s=10, color='#A33EA1')
bins_p = pd.cut(trainers['patience'], bins=15)
means_p = trainers.groupby(bins_p, observed=True)['strategy_score'].mean()
bin_centers_p = [b.mid for b in means_p.index]
ax.plot(bin_centers_p, means_p.values, 'o-', color='#EE1515', linewidth=2, markersize=5)
ax.set_xlabel('Patience (LATENT -- normally hidden!)')
ax.set_ylabel('Strategy Score')
ax.set_title('Patience -> Strategy\n(Unobserved confounder)')

fig.suptitle('Key Causal Relationships in kanto_trainers', fontsize=18, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

### 2.4 Summary statistics

In [ ]:
# Quick descriptive stats for the key variables
key_cols = [
    'wealth', 'trainer_experience', 'strategy_score', 'team_level_avg',
    'type_diversity', 'badges', 'dept_store_spending', 'play_hours',
    'tm_count', 'exp_share_used', 'total_battles_won', 'pokedex_completion',
    'patience', 'natural_talent', 'dedication',
]
trainers[key_cols].describe().round(2)

## 3. Other Key Datasets

Let's generate and inspect the datasets designed for specific causal methods.

### 3.1 Pewter Protein RCT (Chapter 2: Experiments)

An RCT with **noncompliance** -- perfect for practising ITT vs. LATE.

**Structural equations:**
```
treatment_assigned  ~ Bernoulli(0.5)       # random assignment
compliance_type     ~ Categorical(60% complier, 15% always-taker, 25% never-taker)
treatment_received  = f(compliance_type, treatment_assigned)
win_logit           = -3.5 + 0.12*level + 0.01*strategy + type_bonus + 0.7*received
brock_win           ~ Bernoulli(logistic(win_logit))
```

The **true causal effect** of protein is +0.7 on the log-odds scale (~15pp on win probability).

In [ ]:
dgp.RNG = np.random.default_rng(151)  # reset for reproducibility
_ = dgp.generate_kanto_trainers()      # consume the same RNG draws
_ = dgp.generate_kanto_battles()       # consume battle draws
_ = dgp.generate_cities_panel()        # consume city panel draws

protein = dgp.generate_protein_rct(n=200)
print(f"Shape: {protein.shape}")
print(f"\nCompliance type distribution:")
print(protein['compliance_type'].value_counts())
print(f"\nITT (assigned): {protein.groupby('treatment_assigned')['brock_win'].mean().to_dict()}")
print(f"Naive (received): {protein.groupby('treatment_received')['brock_win'].mean().to_dict()}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: Assignment vs. receipt
ax = axes[0]
ct = pd.crosstab(protein['treatment_assigned'], protein['treatment_received'])
ct.plot(kind='bar', ax=ax, color=['#3B4CCA', '#EE1515'])
ax.set_xlabel('Assigned')
ax.set_ylabel('Count')
ax.set_title('Assignment vs. Receipt\n(Noncompliance visible!)')
ax.legend(title='Received', labels=['No', 'Yes'])
ax.tick_params(axis='x', rotation=0)

# Panel 2: Win rate by compliance type
ax = axes[1]
comp_wins = protein.groupby('compliance_type')['brock_win'].mean().sort_values()
bars = ax.barh(comp_wins.index, comp_wins.values, color=['#EE1515', '#3B4CCA', '#4DAD5B'])
ax.set_xlabel('Win Rate vs. Brock')
ax.set_title('Win Rate by Compliance Type')
for bar, val in zip(bars, comp_wins.values):
    ax.text(bar.get_width() + 0.01, bar.get_y() + bar.get_height()/2,
            f'{val:.2f}', va='center', fontweight='bold')

# Panel 3: Type advantage matters!
ax = axes[2]
type_wins = protein.groupby('starter_type')['brock_win'].mean()
colors = [type_color(t.lower()) for t in type_wins.index]
ax.bar(type_wins.index, type_wins.values, color=colors, edgecolor='white')
ax.set_ylabel('Win Rate')
ax.set_title('Type Advantage vs. Brock (Rock)\n(Water, Grass >> Fire)')
for i, (t, v) in enumerate(type_wins.items()):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold')

fig.suptitle('Pewter Protein RCT: Key Patterns', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

### 3.2 Safari Zone Lottery (Chapter 6: Instrumental Variables)

The lottery creates a **valid instrument** (random, affects attendance, no direct effect on battle wins).

Let's verify the **first stage strength** and the **exclusion restriction**.

In [ ]:
# Continue from the same RNG state for the remaining datasets
# In practice, safari_lottery is generated after ss_anne
dgp.RNG = np.random.default_rng(151)
_ = dgp.generate_kanto_trainers()
_ = dgp.generate_kanto_battles()
_ = dgp.generate_cities_panel()
_ = dgp.generate_protein_rct()
_ = dgp.generate_ss_anne()

safari = dgp.generate_safari_lottery(n=600)
print(f"Shape: {safari.shape}")
print(f"\nFirst stage (lottery -> attendance):")
print(safari.groupby('lottery_won')['safari_attended'].mean())
print(f"\nFirst stage difference: {safari.groupby('lottery_won')['safari_attended'].mean().diff().iloc[-1]:.3f}")

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 5))

# Panel 1: First stage -- lottery predicts attendance
ax = axes[0]
first_stage = safari.groupby('lottery_won')['safari_attended'].mean()
ax.bar(['Lost Lottery', 'Won Lottery'], first_stage.values,
       color=['#3B4CCA', '#4DAD5B'], edgecolor='white')
ax.set_ylabel('P(Attended Safari Zone)')
ax.set_title('First Stage: Lottery -> Attendance')
for i, v in enumerate(first_stage.values):
    ax.text(i, v + 0.02, f'{v:.2f}', ha='center', fontweight='bold', fontsize=14)
ax.set_ylim(0, 1.0)

# Panel 2: Reduced form -- lottery -> outcome
ax = axes[1]
reduced_form = safari.groupby('lottery_won')['battle_wins_post'].mean()
ax.bar(['Lost Lottery', 'Won Lottery'], reduced_form.values,
       color=['#3B4CCA', '#4DAD5B'], edgecolor='white')
ax.set_ylabel('Mean Battle Wins (Post)')
ax.set_title('Reduced Form: Lottery -> Outcome')
for i, v in enumerate(reduced_form.values):
    ax.text(i, v + 0.5, f'{v:.1f}', ha='center', fontweight='bold', fontsize=14)

# Panel 3: Compliance types
ax = axes[2]
comp = safari['compliance_type'].value_counts()
ax.pie(comp.values, labels=comp.index, autopct='%1.0f%%',
       colors=['#4DAD5B', '#3B4CCA', '#EE1515'],
       startangle=90, textprops={'fontsize': 12})
ax.set_title('Compliance Types\n(Latent -- driven by patience)')

fig.suptitle('Safari Zone Lottery: IV Diagnostics', fontsize=16, fontweight='bold', y=1.02)
fig.tight_layout()
plt.show()

In [ ]:
# Compute the Wald estimate
fs = safari.groupby('lottery_won')['safari_attended'].mean()
rf = safari.groupby('lottery_won')['battle_wins_post'].mean()

wald_estimate = (rf[1] - rf[0]) / (fs[1] - fs[0])
naive_ols = (
    safari[safari['safari_attended'] == 1]['battle_wins_post'].mean()
    - safari[safari['safari_attended'] == 0]['battle_wins_post'].mean()
)

print(f"True causal effect (from DGP):  +8.0 battle wins")
print(f"Wald IV estimate:               {wald_estimate:+.2f}")
print(f"Naive OLS (biased by patience): {naive_ols:+.2f}")
print(f"\nNotice: OLS overestimates because patient trainers both attend more AND win more.")

### 3.3 Happiness Evolution (Chapter 7: Regression Discontinuity)

The RDD dataset has a sharp threshold at happiness = 220 where Pokemon evolve.

In [ ]:
dgp.RNG = np.random.default_rng(151)
_ = dgp.generate_kanto_trainers()
_ = dgp.generate_kanto_battles()
_ = dgp.generate_cities_panel()
_ = dgp.generate_protein_rct()
_ = dgp.generate_ss_anne()
_ = dgp.generate_safari_lottery()

happiness = dgp.generate_happiness_evolution(n=800)
print(f"Shape: {happiness.shape}")
print(f"\nAbove threshold (>= 220): {happiness['above_threshold'].sum()} / {len(happiness)}")
print(f"Actually evolved: {happiness['evolved'].sum()} (some pressed B!)")
print(f"Pressed B to cancel: {happiness['trainer_pressed_b'].sum()}")

In [ ]:
from kanto_utils import rdd_plot

fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Panel 1: The sharp discontinuity
rdd_plot(
    happiness['happiness_score'].values,
    happiness['battle_performance_post'].values,
    cutoff=220,
    ax=axes[0],
)
axes[0].set_xlabel('Happiness Score')
axes[0].set_ylabel('Battle Performance (Post)')
axes[0].set_title('Sharp RDD: Evolution at Happiness >= 220\n(True effect = +12 points)')

# Panel 2: Density of the running variable
axes[1].hist(happiness['happiness_score'], bins=40, color='#3B4CCA',
             edgecolor='white', alpha=0.7)
axes[1].axvline(220, color='#FFD733', linestyle='--', linewidth=2, label='Cutoff = 220')
axes[1].set_xlabel('Happiness Score')
axes[1].set_ylabel('Count')
axes[1].set_title('Running Variable Density\n(No bunching = no manipulation)')
axes[1].legend()

fig.tight_layout()
plt.show()

## 4. How Changing the DGP Changes Causal Conclusions

One of the most powerful things you can do is **modify the structural equations** and see how your estimates respond. Below we demonstrate three scenarios.

### 4.1 Scenario A: Strengthening a confounder

What if `patience` had a much stronger effect on strategy?

In [ ]:
def generate_trainers_strong_confounder(n=2000, patience_coeff=0.25):
    """Modified DGP where patience has a configurable effect on strategy.
    In the original DGP, the coefficient on patience in the strategy equation is 0.25.
    """
    rng = np.random.default_rng(151)
    
    patience = rng.uniform(0, 100, size=n)
    natural_talent = rng.uniform(0, 100, size=n)
    trainer_experience = np.clip(rng.exponential(4.0, size=n), 0, 15).round(1)
    
    # Strategy with configurable patience coefficient
    strategy_score = np.clip(
        20 + patience_coeff * patience + 0.3 * natural_talent
        + 1.5 * trainer_experience + rng.normal(0, 8, n),
        0, 100,
    ).round(1)
    
    safari_zone_visits = np.clip(
        rng.poisson(1 + 0.04 * patience, size=n), 0, 30
    ).astype(int)
    
    return pd.DataFrame({
        'patience': patience.round(2),
        'strategy_score': strategy_score,
        'safari_zone_visits': safari_zone_visits,
        'trainer_experience': trainer_experience,
    })

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
coefficients = [0.05, 0.25, 0.80]  # weak, original, strong
labels = ['Weak (0.05)', 'Original (0.25)', 'Strong (0.80)']
colors = ['#3B4CCA', '#4DAD5B', '#EE1515']

for ax, coeff, label, color in zip(axes, coefficients, labels, colors):
    df = generate_trainers_strong_confounder(patience_coeff=coeff)
    
    # Naive correlation between safari_zone_visits and strategy
    corr = df['safari_zone_visits'].corr(df['strategy_score'])
    
    ax.scatter(df['safari_zone_visits'], df['strategy_score'],
              alpha=0.15, s=10, color=color)
    ax.set_xlabel('Safari Zone Visits')
    ax.set_ylabel('Strategy Score')
    ax.set_title(f'Patience -> Strategy coeff = {label}\nr = {corr:.3f}')

fig.suptitle('Confounding Strength: How Patience Links Safari Visits to Strategy',
             fontsize=15, fontweight='bold', y=1.03)
fig.tight_layout()
plt.show()

print("Takeaway: As the patience->strategy link strengthens, the spurious")
print("correlation between safari visits and strategy grows. Without controlling")
print("for patience, you'd attribute safari visits' 'effect' on strategy to the wrong cause.")

### 4.2 Scenario B: Removing the treatment effect in the RCT

In [ ]:
def generate_null_rct(n=500, true_effect=0.0):
    """Generate an RCT with a configurable treatment effect.
    Set true_effect=0 to simulate the null hypothesis."""
    rng = np.random.default_rng(42)
    
    treatment = rng.binomial(1, 0.5, n)
    team_level = np.clip(rng.normal(14, 3, n), 5, 25).round(0).astype(int)
    
    win_logit = -3.5 + 0.12 * team_level + true_effect * treatment
    win_prob = 1 / (1 + np.exp(-np.clip(win_logit, -500, 500)))
    win = (rng.uniform(size=n) < win_prob).astype(int)
    
    return pd.DataFrame({
        'treatment': treatment,
        'team_level': team_level,
        'win': win,
    })

fig, axes = plt.subplots(1, 3, figsize=(16, 5))
effects = [0.0, 0.7, 2.0]
effect_labels = ['No effect (0.0)', 'Original (0.7)', 'Strong (2.0)']

for ax, effect, label in zip(axes, effects, effect_labels):
    df = generate_null_rct(n=500, true_effect=effect)
    rates = df.groupby('treatment')['win'].mean()
    diff = rates[1] - rates[0]
    
    ax.bar(['Control', 'Treated'], rates.values,
           color=['#3B4CCA', '#EE1515'], edgecolor='white')
    ax.set_ylabel('Win Rate')
    ax.set_title(f'True log-odds effect = {label}\nEstimated diff = {diff:+.3f}')
    ax.set_ylim(0, 1.0)
    for i, v in enumerate(rates.values):
        ax.text(i, v + 0.02, f'{v:.3f}', ha='center', fontweight='bold')

fig.suptitle('Varying the True Treatment Effect', fontsize=15, fontweight='bold', y=1.03)
fig.tight_layout()
plt.show()

### 4.3 Scenario C: Breaking the instrument

In [ ]:
def generate_broken_iv(n=600, direct_lottery_effect=0.0):
    """Safari lottery where the instrument may violate exclusion.
    If direct_lottery_effect > 0, the lottery has a direct effect on outcomes
    (violating the exclusion restriction)."""
    rng = np.random.default_rng(151)
    
    patience = rng.uniform(0, 100, n)
    lottery_won = rng.binomial(1, 0.5, n)
    
    # Treatment (attendance) driven by lottery + patience
    attend_prob = 1 / (1 + np.exp(-(-1.0 + 2.0 * lottery_won + 0.02 * patience)))
    attended = (rng.uniform(size=n) < attend_prob).astype(int)
    
    # Outcome: true safari effect = 8, but lottery may have direct effect
    battle_wins = np.clip(
        30 + 8.0 * attended + 0.15 * patience
        + direct_lottery_effect * lottery_won  # exclusion violation!
        + rng.normal(0, 6, n),
        0, 100,
    ).round(0).astype(int)
    
    # Compute Wald estimate
    fs_diff = attended[lottery_won == 1].mean() - attended[lottery_won == 0].mean()
    rf_diff = battle_wins[lottery_won == 1].mean() - battle_wins[lottery_won == 0].mean()
    wald = rf_diff / fs_diff if abs(fs_diff) > 0.01 else float('nan')
    
    return wald, fs_diff

# Test different exclusion-restriction violations
direct_effects = np.linspace(0, 15, 30)
wald_estimates = []
for de in direct_effects:
    w, _ = generate_broken_iv(direct_lottery_effect=de)
    wald_estimates.append(w)

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot(direct_effects, wald_estimates, 'o-', color='#EE1515', linewidth=2)
ax.axhline(8.0, color='#4DAD5B', linestyle='--', linewidth=2, label='True effect = 8.0')
ax.axvline(0.0, color='#FFD733', linestyle=':', linewidth=2, label='Exclusion holds')
ax.set_xlabel('Direct Effect of Lottery on Outcome (Exclusion Violation)', fontsize=13)
ax.set_ylabel('Wald IV Estimate', fontsize=13)
ax.set_title('What Happens When You Break the Exclusion Restriction?', fontsize=15)
ax.legend(fontsize=12)
plt.tight_layout()
plt.show()

print("When the lottery has NO direct effect on outcomes (exclusion holds), the Wald")
print("estimate is close to the true effect of 8.0. As we add a direct effect,")
print("the IV estimate becomes biased -- the instrument is 'broken'.")

## 5. Exercise: Modify a Structural Equation

Now it's your turn! Modify the code below to explore how the DGP controls causal conclusions.

In [ ]:
nurse_joy_says(
    "Try changing the parameters below and re-running the cell. "
    "Can you find a set of parameters where the naive estimate is <em>more</em> "
    "biased than the original? What about a set where the bias disappears?"
)

In [ ]:
# =================================================================
# EXERCISE: Modify these parameters and re-run!
# =================================================================

# --- Your parameters ---
N_TRAINERS = 600           # Sample size
TRUE_SAFARI_EFFECT = 8.0   # True causal effect of Safari Zone (try 0, 5, 20)
PATIENCE_ON_OUTCOME = 0.15 # Confounding: patience -> battle wins (try 0, 0.5, 1.0)
PATIENCE_ON_ATTEND = 0.04  # Confounding: patience -> attendance (try 0, 0.1, 0.2)

# --- Generate data with your parameters ---
rng = np.random.default_rng(42)
patience = rng.uniform(0, 100, N_TRAINERS)
lottery_won = rng.binomial(1, 0.5, N_TRAINERS)

# Attendance depends on lottery + patience
attend_logit = -1.5 + 1.5 * lottery_won + PATIENCE_ON_ATTEND * patience
attend_prob = 1 / (1 + np.exp(-np.clip(attend_logit, -500, 500)))
attended = (rng.uniform(size=N_TRAINERS) < attend_prob).astype(int)

# Outcome
battle_wins = np.clip(
    30 + TRUE_SAFARI_EFFECT * attended
    + PATIENCE_ON_OUTCOME * patience
    + rng.normal(0, 6, N_TRAINERS),
    0, 100
).round(0).astype(int)

# --- Estimates ---
naive = battle_wins[attended == 1].mean() - battle_wins[attended == 0].mean()

fs = attended[lottery_won == 1].mean() - attended[lottery_won == 0].mean()
rf = battle_wins[lottery_won == 1].mean() - battle_wins[lottery_won == 0].mean()
iv_est = rf / fs if abs(fs) > 0.01 else float('nan')

print("=" * 50)
print(f"  True causal effect:      {TRUE_SAFARI_EFFECT:+.1f}")
print(f"  Naive OLS estimate:      {naive:+.2f}  (bias = {naive - TRUE_SAFARI_EFFECT:+.2f})")
print(f"  IV / Wald estimate:      {iv_est:+.2f}  (bias = {iv_est - TRUE_SAFARI_EFFECT:+.2f})")
print(f"  First stage strength:    {fs:.3f}")
print("=" * 50)
print()
if abs(naive - TRUE_SAFARI_EFFECT) > abs(iv_est - TRUE_SAFARI_EFFECT):
    print("The IV estimate is closer to the truth -- it's handling the confounding!")
else:
    print("Hmm, the IV isn't doing better here. Check the first stage strength.")

In [ ]:
badge_earned("Boulder", chapter=0)
print("\nYou've completed Appendix B! You now understand how every dataset")
print("in this textbook was generated and can modify the DGP to test your intuitions.")